# 비용 청구 분석

이 노트북은 플러그인을 사용하여 여행 경비를 지역 영수증 이미지에서 처리하고, 비용 청구 이메일을 생성하며, 파이 차트를 사용해 경비 데이터를 시각화하는 에이전트를 만드는 방법을 보여줍니다. 에이전트는 작업 상황에 따라 동적으로 기능을 선택합니다.

단계:
1. OCR 에이전트가 지역 영수증 이미지를 처리하고 여행 경비 데이터를 추출합니다.
2. 이메일 에이전트가 비용 청구 이메일을 생성합니다.

### 여행 경비 시나리오 예시:
당신이 다른 도시로 출장 가는 직원이라고 상상해보세요. 회사는 모든 합리적인 여행 관련 경비를 환급해 주는 정책을 가지고 있습니다. 잠재적인 여행 경비는 다음과 같습니다:
- 교통비:
왕복 항공권 귀하의 거주 도시에서 목적지 도시까지.
공항까지 및 공항에서 택시 또는 라이드 헤일링 서비스.
목적지 도시 내의 지역 교통수단(대중교통, 렌터카, 택시 등).

- 숙박비:
회의 장소 근처 중간급 비즈니스 호텔에서 3박 숙박.

- 식비:
회사의 일일 식비 정책에 따라 아침, 점심, 저녁 일일 식대.

- 기타 비용:
공항 주차 요금.
호텔 인터넷 접속 요금.
팁 또는 소규모 서비스 요금.

- 문서화:
모든 영수증(항공권, 택시, 호텔, 식사 등)과 완성된 비용 보고서를 환급을 위해 제출합니다.


## 필요한 라이브러리 가져오기

노트북에 필요한 라이브러리와 모듈을 가져옵니다.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## 비용 모델 정의

개별 비용에 대한 Pydantic 모델과 사용자 쿼리를 구조화된 비용 데이터로 변환하는 ExpenseFormatter 클래스를 만듭니다.

각 비용은 다음 형식으로 표시됩니다:  
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## 도구 정의 - 이메일 생성하기

비용 청구서를 제출하기 위한 이메일을 생성하는 도구 함수를 만드세요.
- 이 도구는 Microsoft Agent Framework의 `@tool` 데코레이터를 사용합니다.
- 비용의 총액을 계산하고 세부 정보를 이메일 본문 형식으로 만듭니다.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# 영수증 이미지에서 여행 경비를 추출하는 도구

영수증 이미지에서 여행 경비를 추출하는 도구 함수를 만드세요.
- 이 도구는 Microsoft Agent Framework의 `@tool` 데코레이터를 사용합니다.
- 영수증 이미지를 읽고, base64로 인코딩하며, 에이전트가 분석할 수 있도록 데이터 URI를 반환합니다.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## 비용 처리

에이전트를 정의하고 `WorkflowBuilder`를 사용하여 순차적인 워크플로우에 연결합니다.
- OCR 에이전트는 `load_receipt_image` 도구를 사용하여 영수증 이미지에서 구조화된 비용 데이터를 추출합니다.
- 이메일 에이전트는 추출된 데이터를 가져와 `generate_expense_email` 도구를 사용하여 전문적인 비용 청구 이메일을 생성합니다.
- `add_edge`를 사용한 `WorkflowBuilder`는 순차 파이프라인을 생성합니다: OCR 에이전트 → 이메일 에이전트.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

순차적인 워크플로를 구성하고 실행하여 영수증 이미지를 처리하고 비용 청구 이메일을 생성합니다.


> **참고:** 이 워크플로우는 현재 영수증 이미지를 base64 텍스트로 전달하며, 대부분의 채팅 모델(예: gpt-4o)은 이를 이미지로 인식하지 않습니다.
> 또한 모델의 컨텍스트 창을 초과할 수 있습니다. Azure AI Vision(또는 다른 OCR 도구)으로 OCR을 실행하여 추출된 텍스트만 전달하거나 이미지를 `image_url` 메시지로 보내도록 리팩터링하는 것이 좋습니다.
> 컨텍스트 오류를 피하려면 더 작은 영수증 이미지나 더 큰 컨텍스트 창을 가진 모델을 사용해 보세요.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
